In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

torch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
VRAM: 14.6 GB


In [ ]:
!pip install -q \
    "transformers<5" \
    "sentence-transformers" \
    "accelerate" \
    "datasets" \
    "huggingface_hub"

In [ ]:
import torch
import transformers
import sentence_transformers

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

from sentence_transformers import SentenceTransformer, InputExample, losses

print("OK")

torch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
transformers: 4.57.6
sentence-transformers: 5.7.0
OK


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

print(whoami()["name"])

Schamanichands


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset

dataset = load_dataset("hreyulog/arkts-code-docstring")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['nwo', 'sha', 'path', 'language', 'identifier', 'docstring', 'function', 'ast_function', 'obf_function', 'url', 'function_sha', 'source'],
        num_rows: 19561
    })
    validation: Dataset({
        features: ['nwo', 'sha', 'path', 'language', 'identifier', 'docstring', 'function', 'ast_function', 'obf_function', 'url', 'function_sha', 'source'],
        num_rows: 2445
    })
    test: Dataset({
        features: ['nwo', 'sha', 'path', 'language', 'identifier', 'docstring', 'function', 'ast_function', 'obf_function', 'url', 'function_sha', 'source'],
        num_rows: 2446
    })
})


In [ ]:
print(dataset["train"].column_names)
print(dataset["train"][0])

['nwo', 'sha', 'path', 'language', 'identifier', 'docstring', 'function', 'ast_function', 'obf_function', 'url', 'function_sha', 'source']
{'nwo': 'openharmony-tpc/ohos_mpchart', 'sha': '4fb43a7137320ef2fee2634598f53d93975dfb4a', 'path': 'library/src/main/ets/components/components/AxisBase.ets', 'language': 'arkts', 'identifier': 'setAxisMaxLabels', 'docstring': 'The maximum number of labels on the axis', 'function': 'public setAxisMaxLabels(labels: number): void {\n    if (labels > 0)\n      this.mAxisMaxLabels = labels;\n  }', 'ast_function': 'AST#method_declaration#Left public setAxisMaxLabels AST#parameter_list#Left ( AST#parameter#Left labels : AST#type_annotation#Left AST#primary_type#Left number AST#primary_type#Right AST#type_annotation#Right AST#parameter#Right ) AST#parameter_list#Right : AST#type_annotation#Left AST#primary_type#Left void AST#primary_type#Right AST#type_annotation#Right AST#block_statement#Left { AST#statement#Left AST#if_statement#Left if ( AST#expression#L

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "google/embeddinggemma-300m"

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda"
)

model.max_seq_length = 512

print("Model loaded")
print("Device:", model.device)
print("Max sequence length:", model.max_seq_length)

Model loaded
Device: cuda:0
Max sequence length: 512


In [ ]:
import gc
import torch
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, losses

# Освободить VRAM после неудачной попытки batch=16
gc.collect()
torch.cuda.empty_cache()

# Создаём training examples заново
train_examples = [
    InputExample(
        texts=[row["docstring"], row["function"]]
    )
    for row in dataset["train"]
]

# Batch 8 — он уже успешно прошёл тест
BATCH_SIZE = 4

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE
)

train_loss = losses.MultipleNegativesRankingLoss(model)

print("Train examples:", len(train_examples))
print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", len(train_dataloader))
print(
    "GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Train examples: 19561
Batch size: 4
Steps per epoch: 4891
GPU memory: 1.15 GB


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/arkts-nir/embeddinggemma-300m-arkts"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(OUTPUT_DIR)

/content/drive/MyDrive/arkts-nir/embeddinggemma-300m-arkts


In [ ]:
import gc
import torch
from sentence_transformers import SentenceTransformer, losses

gc.collect()
torch.cuda.empty_cache()

model = SentenceTransformer(
    "google/embeddinggemma-300m",
    device="cuda"
)
model.max_seq_length = 512

train_loss = losses.MultipleNegativesRankingLoss(model)

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

GPU: Tesla T4
VRAM: 14.56317138671875


In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(len(train_dataloader) * 2 * 0.1),
    optimizer_params={"lr": 2e-5},
    use_amp=True,
    show_progress_bar=True,
    output_path=OUTPUT_DIR
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.113200
1000,0.199500
1500,0.297700
2000,0.326900
2500,0.301500
3000,0.274100
3500,0.306400
4000,0.207100
4500,0.190800
5000,0.172900


In [ ]:
import os

print(os.path.exists(OUTPUT_DIR))
print(os.listdir(OUTPUT_DIR))

True
['config_sentence_transformers.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'tokenizer.model', 'tokenizer.json', 'sentence_bert_config.json', '1_Pooling', '2_Dense', '3_Dense', '4_Normalize', 'modules.json', 'README.md']


In [ ]:
from sentence_transformers import SentenceTransformer

loaded_model = SentenceTransformer(
    OUTPUT_DIR,
    device="cuda"
)

loaded_model.max_seq_length = 512

print("Model loaded successfully")
print("Device:", loaded_model.device)
print("Max sequence length:", loaded_model.max_seq_length)

The tokenizer you are loading from '/content/drive/MyDrive/arkts-nir/embeddinggemma-300m-arkts' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model loaded successfully
Device: cuda:0
Max sequence length: 512


In [ ]:
from datasets import load_dataset

dataset = load_dataset("hreyulog/arkts-code-docstring")

test_data = dataset["test"]

print("Test examples:", len(test_data))

Test examples: 2446


In [ ]:
import numpy as np
import torch

queries = [row["docstring"] for row in test_data]
documents = [row["function"] for row in test_data]

print("Queries:", len(queries))
print("Documents:", len(documents))

query_embeddings = loaded_model.encode(
    queries,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

document_embeddings = loaded_model.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Query embeddings:", query_embeddings.shape)
print("Document embeddings:", document_embeddings.shape)

Queries: 2446
Documents: 2446


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Query embeddings: (2446, 768)
Document embeddings: (2446, 768)


In [ ]:
import numpy as np

# Cosine similarity, поскольку embeddings были normalized
similarity = query_embeddings @ document_embeddings.T

# Для каждого query получаем ранжированный список документов
rankings = np.argsort(-similarity, axis=1)

ranks = []

for i in range(len(test_data)):
    # Правильный документ для query i находится на позиции i
    rank = np.where(rankings[i] == i)[0][0] + 1
    ranks.append(rank)

ranks = np.array(ranks)

# Recall@K
def recall_at_k(ranks, k):
    return np.mean(ranks <= k)

# MRR
def mean_reciprocal_rank(ranks):
    return np.mean(1.0 / ranks)

# NDCG@K для одного релевантного документа
def ndcg_at_k(ranks, k):
    scores = np.where(
        ranks <= k,
        1.0 / np.log2(ranks + 1),
        0.0
    )
    return np.mean(scores)

results = {
    "R@1": recall_at_k(ranks, 1),
    "R@5": recall_at_k(ranks, 5),
    "MRR": mean_reciprocal_rank(ranks),
    "NDCG@5": ndcg_at_k(ranks, 5),
}

print("Fine-tuned EmbeddingGemma-300M")
print("-" * 40)

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Fine-tuned EmbeddingGemma-300M
----------------------------------------
R@1: 0.5576
R@5: 0.7690
MRR: 0.6529
NDCG@5: 0.6730


In [ ]:
from datasets import Dataset

train_full = dataset["train"]

train_25 = train_full.shuffle(seed=42).select(
    range(int(len(train_full) * 0.25))
)

train_50 = train_full.shuffle(seed=42).select(
    range(int(len(train_full) * 0.50))
)

print("100%:", len(train_full))
print("50% :", len(train_50))
print("25% :", len(train_25))

100%: 19561
50% : 9780
25% : 4890


In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, losses

train_examples_25 = [
    InputExample(texts=[row["docstring"], row["function"]])
    for row in train_25
]

train_dataloader_25 = DataLoader(
    train_examples_25,
    shuffle=True,
    batch_size=4,
    collate_fn=loaded_model.smart_batching_collate
)

print("Training examples:", len(train_examples_25))
print("Batches per epoch:", len(train_dataloader_25))

Training examples: 4890
Batches per epoch: 1223


In [ ]:
import gc
import torch
from sentence_transformers import SentenceTransformer, losses

gc.collect()
torch.cuda.empty_cache()

model_25 = SentenceTransformer(
    "google/embeddinggemma-300m",
    device="cuda"
)
model_25.max_seq_length = 512

train_loss_25 = losses.MultipleNegativesRankingLoss(model_25)

print("Model:", "google/embeddinggemma-300m")
print("Device:", model_25.device)
print("Max sequence length:", model_25.max_seq_length)

Model: google/embeddinggemma-300m
Device: cuda:0
Max sequence length: 512


In [ ]:
OUTPUT_DIR_25 = "/content/drive/MyDrive/arkts-nir/embeddinggemma-300m-arkts-25"

model_25.fit(
    train_objectives=[(train_dataloader_25, train_loss_25)],
    epochs=2,
    warmup_steps=int(len(train_dataloader_25) * 2 * 0.1),
    optimizer_params={"lr": 2e-5},
    use_amp=True,
    show_progress_bar=True,
    output_path=OUTPUT_DIR_25,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.214300
1000,0.353000
1500,0.000000
2000,0.000000


In [ ]:
import gc
import torch

del model_25
del model_25_eval
del train_loss_25

gc.collect()
torch.cuda.empty_cache()

print("CUDA cache cleared")

CUDA cache cleared


In [ ]:
model_25 = SentenceTransformer(
    "google/embeddinggemma-300m",
    device="cuda"
)

model_25.max_seq_length = 512

print("Model loaded successfully")
print("Device:", model_25.device)
print("Max sequence length:", model_25.max_seq_length)

Model loaded successfully
Device: cuda:0
Max sequence length: 512


In [ ]:
test_query = model_25.encode(
    [test_data[0]["docstring"]],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

test_document = model_25.encode(
    [test_data[0]["function"]],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Query has NaN:", np.isnan(test_query).any())
print("Document has NaN:", np.isnan(test_document).any())

print("Query norm:", np.linalg.norm(test_query[0]))
print("Document norm:", np.linalg.norm(test_document[0]))

Query has NaN: False
Document has NaN: False
Query norm: 1.0
Document norm: 1.0


In [ ]:
train_loss_25 = losses.MultipleNegativesRankingLoss(model_25)

In [ ]:
OUTPUT_DIR_25_SAFE = "/content/drive/MyDrive/arkts-nir/embeddinggemma-300m-arkts-25-safe"

model_25.fit(
    train_objectives=[(train_dataloader_25, train_loss_25)],
    epochs=2,
    warmup_steps=int(len(train_dataloader_25) * 2 * 0.1),
    optimizer_params={"lr": 1e-5},
    use_amp=False,
    show_progress_bar=True,
    output_path=OUTPUT_DIR_25_SAFE,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.145000
1000,0.142500
1500,0.091000
2000,0.053300


In [ ]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

print("Saved:", os.path.exists(OUTPUT_DIR_25_SAFE))

model_25_safe = SentenceTransformer(
    OUTPUT_DIR_25_SAFE,
    device="cuda"
)
model_25_safe.max_seq_length = 512

check_query = model_25_safe.encode(
    [test_data[0]["docstring"]],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

check_document = model_25_safe.encode(
    [test_data[0]["function"]],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Query has NaN:", np.isnan(check_query).any())
print("Document has NaN:", np.isnan(check_document).any())
print("Query norm:", np.linalg.norm(check_query[0]))
print("Document norm:", np.linalg.norm(check_document[0]))

Saved: True


The tokenizer you are loading from '/content/drive/MyDrive/arkts-nir/embeddinggemma-300m-arkts-25-safe' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Query has NaN: False
Document has NaN: False
Query norm: 1.0
Document norm: 1.0


In [ ]:
queries = [row["docstring"] for row in test_data]
documents = [row["function"] for row in test_data]

query_embeddings_25_safe = model_25_safe.encode(
    queries,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

document_embeddings_25_safe = model_25_safe.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Query embeddings:", query_embeddings_25_safe.shape)
print("Document embeddings:", document_embeddings_25_safe.shape)

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Query embeddings: (2446, 768)
Document embeddings: (2446, 768)


In [ ]:
similarity_25_safe = query_embeddings_25_safe @ document_embeddings_25_safe.T

rankings_25_safe = np.argsort(-similarity_25_safe, axis=1)

ranks_25_safe = np.array([
    np.where(rankings_25_safe[i] == i)[0][0] + 1
    for i in range(len(test_data))
])

results_25_safe = {
    "R@1": recall_at_k(ranks_25_safe, 1),
    "R@5": recall_at_k(ranks_25_safe, 5),
    "MRR": mean_reciprocal_rank(ranks_25_safe),
    "NDCG@5": ndcg_at_k(ranks_25_safe, 5),
}

print("Fine-tuned EmbeddingGemma-300M — 25% training data")
print("-" * 50)

for metric, value in results_25_safe.items():
    print(f"{metric}: {value:.4f}")

Fine-tuned EmbeddingGemma-300M — 25% training data
--------------------------------------------------
R@1: 0.6390
R@5: 0.8115
MRR: 0.7175
NDCG@5: 0.7338


In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import math